In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [8]:
train = pd.read_csv('public_trip_data.csv')
test = pd.read_csv('private_trip_data.csv')

In [9]:
train_id = train["TripID"]
test_id = test['TripID']

In [10]:
y = train['HighCarbon']
x = train.drop(columns = ['HighCarbon','TripID',"EmployeeNumber",
"Departure_CO2e","Return_CO2e","Hotel_CO2e","Spend_CO2e","TotalCO2e","HighCarbon"])
test = test.drop(columns = ['TripID'])


In [11]:
cat_cols = x.select_dtypes(include="object").columns.tolist()

print(cat_cols)

['DepartureLocationCountry', 'DepartureLocationCity', 'ArrivalLocationCountry', 'ArrivalLocationCity', 'ShippingTypeDescription', 'Purpose', 'OutOfPolicy', 'BusinessUnit']


C:\Users\koush\AppData\Local\Temp\ipykernel_16268\597484544.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = x.select_dtypes(include="object").columns.tolist()


In [12]:
event_log = pd.read_csv("public_trip_event_log.csv")
event_features = event_log.groupby("TripID").agg(
    EventCount=("EventName","count"),
    UniqueEvents=("EventName","nunique"),
    FirstStep=("StepOrder","min"),
    LastStep=("StepOrder","max")
).reset_index()
train = train.merge(event_features,on="TripID",how="left")


In [13]:
private_log = pd.read_csv("private_trip_event_log.csv")

private_features = private_log.groupby("TripID").agg(
    EventCount=("EventName","count"),
    UniqueEvents=("EventName","nunique"),
    FirstStep=("StepOrder","min"),
    LastStep=("StepOrder","max")
).reset_index()

test = pd.read_csv("private_trip_data.csv")

test = test.merge(private_features,on="TripID",how="left")


In [14]:
y = train["HighCarbon"]

drop_cols = [
    "TripID",
    "EmployeeNumber",
    "Departure_CO2e",
    "Return_CO2e",
    "Hotel_CO2e",
    "Spend_CO2e",
    "TotalCO2e",
    "HighCarbon"
]

X = train.drop(columns=drop_cols)

test_final = test.drop(columns=["TripID"])
cat_cols = X.select_dtypes(include="object").columns.tolist()

print(cat_cols)

['DepartureLocationCountry', 'DepartureLocationCity', 'ArrivalLocationCountry', 'ArrivalLocationCity', 'ShippingTypeDescription', 'Purpose', 'OutOfPolicy', 'BusinessUnit']


C:\Users\koush\AppData\Local\Temp\ipykernel_16268\2056025937.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns.tolist()


In [15]:
cat_features = [X.columns.get_loc(col) for col in cat_cols]

print(cat_features)

[0, 1, 2, 3, 5, 6, 7, 9]


In [16]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [17]:
model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=50,
    early_stopping_rounds=50
)
model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_valid, y_valid),
    use_best_model=True
)

0:	test: 0.9930502	best: 0.9930502 (0)	total: 180ms	remaining: 54s
50:	test: 0.9994248	best: 0.9994248 (50)	total: 3.45s	remaining: 16.9s
100:	test: 0.9993574	best: 0.9994248 (50)	total: 6.66s	remaining: 13.1s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9994248322
bestIteration = 50

Shrink model to first 51 iterations.


CatBoostClassifier(depth=6, early_stopping_rounds=50, eval_metric='AUC', iterations=300, learning_rate=0.1, loss_function='Logloss', random_seed=42, verbose=50)

In [18]:
from sklearn.metrics import roc_auc_score

pred_prob = model.predict_proba(X_valid)[:, 1]

roc = roc_auc_score(y_valid, pred_prob)

print("ROC-AUC:", roc)
test_prob = model.predict_proba(test_final)[:, 1]

ROC-AUC: 0.9994248321573572


In [19]:
from sklearn.metrics import classification_report

pred = model.predict(X_valid)

print(classification_report(y_valid, pred))
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_valid, pred))

              precision    recall  f1-score   support

           0       0.99      1.00      1.00      9793
           1       0.99      0.98      0.99      3265

    accuracy                           0.99     13058
   macro avg       0.99      0.99      0.99     13058
weighted avg       0.99      0.99      0.99     13058

[[9763   30]
 [  55 3210]]


In [20]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

auc_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=300,
        learning_rate=0.1,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False,
        early_stopping_rounds=50,
        thread_count=-1
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_val, y_val),
        use_best_model=True
    )

    pred_prob = model.predict_proba(X_val)[:, 1]

    auc = roc_auc_score(y_val, pred_prob)

    auc_scores.append(auc)

    print(f"Fold {fold}: ROC-AUC = {auc:.6f}")

print("\nAverage ROC-AUC:", np.mean(auc_scores))
print("Std Dev:", np.std(auc_scores))

Fold 1: ROC-AUC = 0.999312
Fold 2: ROC-AUC = 0.999342
Fold 3: ROC-AUC = 0.999498
Fold 4: ROC-AUC = 0.999449
Fold 5: ROC-AUC = 0.999361

Average ROC-AUC: 0.9993921777226348
Std Dev: 6.964236852518448e-05


In [21]:
model.fit(
    X,
    y,
    cat_features=cat_features
)

CatBoostClassifier(depth=6, early_stopping_rounds=50, eval_metric='AUC', iterations=300, learning_rate=0.1, loss_function='Logloss', random_seed=42, verbose=False)